Question 5 - Creativity Task

In [ ]:

# HYBRID MULTI-RESOLUTION CNN-MLP ARCHITECTURE

import Modified_code as orig
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# Bind to the centralized computing hardware device target from your original script
device = orig.device

#  ARCHITECTURE DEFINITION

class MultiResolutionHybridNet(nn.Module):
    """
    A Multi-Resolution Hybrid Network that processes inputs at two spatial 
    scales simultaneously using parallel CNN streams, fused by an MLP head.
    """
    def __init__(self):
        super().__init__()
        
        # --- Stream A: High-Resolution CNN Branch (Input Canvas: 32x32) ---
        self.high_res_conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.high_res_conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        # Dimensions after two MaxPools: 32x32 -> 16x16 -> 8x8
        self.high_res_flat_dim = 64 * 8 * 8 
        
        # --- Stream B: Low-Resolution CNN Branch (Input Canvas: 16x16) ---
        self.low_res_conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.low_res_conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        # Dimensions after two MaxPools: 16x16 -> 8x8 -> 4x4
        self.low_res_flat_dim = 32 * 4 * 4
        
        # --- Joint MLP Classification Head ---
        # Combines the features extracted from both resolution branches
        self.fusion_dim = self.high_res_flat_dim + self.low_res_flat_dim
        self.mlp_fc1 = nn.Linear(self.fusion_dim, 256)
        self.mlp_fc2 = nn.Linear(256, 128)
        self.mlp_out = nn.Linear(128, 10) # 10-Class CIFAR-10 Output Matrix

    def forward(self, x):
        # Stream A: Process original High-Resolution representation (32x32)
        h_out = F.max_pool2d(torch.relu(self.high_res_conv1(x)), 2)
        h_out = F.max_pool2d(torch.relu(self.high_res_conv2(h_out)), 2)
        h_flat = h_out.view(h_out.size(0), -1)
        
        # Dynamic Downsampling: Reduce spatial boundaries to Low-Resolution (16x16)
        x_low = F.interpolate(x, size=(16, 16), mode='bilinear', align_corners=False)
        
        # Stream B: Process structural macroeconomic features at Low-Resolution
        l_out = F.max_pool2d(torch.relu(self.low_res_conv1(x_low)), 2)
        l_out = F.max_pool2d(torch.relu(self.low_res_conv2(l_out)), 2)
        l_flat = l_out.view(l_out.size(0), -1)
        
        # Fusion Step: Concatenate structural outputs from both spatial channels
        fused_features = torch.cat((h_flat, l_flat), dim=1)
        
        # Multi-Layer Perceptron Classification Head Execution
        mlp_hidden = torch.relu(self.mlp_fc1(fused_features))
        mlp_hidden = torch.relu(self.mlp_fc2(mlp_hidden))
        return self.mlp_out(mlp_hidden)


# 2. DATA PIPELINES RESTORATION
 

# Hook directly into the standard 10-class dataset loaders initialized in your script
train_loader = orig.train_loader
test_loader = orig.test_loader


# 3. OPTIMIZATION AND INTEGRATED TRAINING LOOP

# Initialize core network modules
mr_hybrid_model = MultiResolutionHybridNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer_mr = optim.Adam(mr_hybrid_model.parameters(), lr=0.001)

num_epochs = 15
mr_train_losses, mr_test_losses = [], []
mr_train_accs, mr_test_accs = [], []

print("Training Custom Dual-Branch Multi-Resolution Architecture...")
print("-" * 80)

for epoch in range(num_epochs):
    # CRITICAL HOOK: Point your original script's tracking variables onto our active optimizer
    orig.optimizer = optimizer_mr
    
    # Safely utilize your shared script utilities for running training iterations
    t_loss, t_acc, _ = orig.train(mr_hybrid_model, train_loader)
    v_loss, v_acc = orig.test(mr_hybrid_model, test_loader)
    
    mr_train_losses.append(t_loss)
    mr_test_losses.append(v_loss)
    mr_train_accs.append(t_acc)
    mr_test_accs.append(v_acc)
    
    print(f"Epoch [{epoch+1:02d}/{num_epochs}] -> "
          f"Train Loss: {t_loss:.4f} | Test Loss: {v_loss:.4f} | "
          f"Train Acc: {t_acc:.2f}% | Test Acc: {v_acc:.2f}%")

print("-" * 80)
print("Multi-Resolution Hybrid Model training phase completed successfully.")


# 4. FIXED COMPREHENSIVE PERFORMANCE PLOT GENERATION

epochs_range = range(1, num_epochs + 1)
baseline_epochs = range(1, len(orig.train_losses) + 1)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

#  PANEL 1: CROSS-MODEL EVALUATION LOSS CURVES 
axes[0].plot(epochs_range, mr_test_losses, label='Multi-Resolution Hybrid', color='#2ca02c', linewidth=2.5)
axes[0].plot(baseline_epochs, orig.test_losses, label='Standard CNN (Baseline)', color='#1f77b4', linewidth=2.5)
if hasattr(orig, 'mlp_test_losses'):
    axes[0].plot(baseline_epochs, orig.mlp_test_losses, label='Standard MLP (Baseline)', color='#ff7f0e', linewidth=2.5)

axes[0].set_title('Figure 19: Cross-Architectural Evaluation Loss Profiles', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss Metric Value')
axes[0].grid(True, linestyle=':', alpha=0.6)
axes[0].legend()

#  PANEL 2: CROSS-MODEL EVALUATION ACCURACY CURVES
axes[1].plot(epochs_range, mr_test_accs, label='Multi-Resolution Hybrid', color='#2ca02c', linewidth=2.5)
axes[1].plot(baseline_epochs, orig.test_accs, label='Standard CNN (Baseline)', color='#1f77b4', linewidth=2.5)
if hasattr(orig, 'mlp_test_accs'):
    axes[1].plot(baseline_epochs, orig.mlp_test_accs, label='Standard MLP (Baseline)', color='#ff7f0e', linewidth=2.5)

axes[1].set_title('Figure 20: Cross-Architectural Test Accuracy Trends (%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Validation Generalization Accuracy (%)')
axes[1].grid(True, linestyle=':', alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()